# TabDPT Classifier Artifact Inference — DIMER tutorial

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `cef1f0ae4af14c7c27364a1f1a9d8f573af3504e`

This notebook consumes an **externally supplied** DIMER v3 artifact and genuinely new unlabelled data. **By the end of this notebook you will be able to:** validate artifact provenance/integrity, reconstruct serving state without refitting preprocessing, score new data, and export predictions/provenance.

References: [README](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/README.md), [model card](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/MODEL_CARD.md), [dataset spec](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/TABULAR_CLASSIFICATION_DATASET_SPEC.md), [DIMER contract](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/DIMER_CONTRACT.md), [pinned upstream](https://github.com/layer6ai-labs/TabDPT-inference/tree/9cfb05e0a6bc380ae6c99c08adc8d50dacd4f246), [Layer6/TabDPT](https://huggingface.co/Layer6/TabDPT), [paper](https://arxiv.org/abs/2410.18164).

## Prerequisites and trust boundary

**Google Colab is the supported release path**, Python 3.11–3.13. Other Jupyter environments may work but are outside this conformance claim. The pinned code SHA is retained by `anchors/notebook-spec-v1-code-20260910`. `use_flash=False` is the portable path. Uploaded artifact/input data remains inside the Colab runtime and is not sent by this notebook to an external inference service; do not upload confidential, restricted, or sensitive data unless that Colab environment is authorized for it. Seeded ensemble/context selection improves repeatability, but device, CUDA/library builds, and kernels may still cause numeric variation.

A matching JSON manifest, size, and SHA-256 establish internal consistency, not sender authenticity. The artifact is data-only but its support table may be governed data. The base checkpoint is separately acquired by immutable identity and digest. This notebook accepts individual files, not archives.

In [ ]:
import sys
if "torch" in sys.modules: raise RuntimeError("Start from a fresh Google Colab runtime before installing dependencies.")
REPO_REVISION="cef1f0ae4af14c7c27364a1f1a9d8f573af3504e"; REPO_DIR="/content/tabdpt-classifier-pipeline"
!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"

## 1. Verify runtime and expected model identity

Report runtime identity and verify the immutable checkpoint before artifact reconstruction. The Hugging Face model repository supplies data files only; model-repository remote Python code is not executed. Successful verification proves expected model-byte identity, not model quality.

In [ ]:
import importlib.metadata as md, platform, torch
from tabdpt_classifier_pipeline import *
print("Python",sys.version.split()[0],"torch",md.version("torch"),"tabdpt",md.version("tabdpt"))
print("Device",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","CUDA",torch.version.cuda)
print("Assumptions: compile_model=False use_flash=False quantization=None precision=framework/device default")
weights=resolve_tabdpt_weights()
print(TABDPT_HF_REPO,TABDPT_HF_REVISION,TABDPT_WEIGHT_SHA256,TABDPT_UPSTREAM_CODE_COMMIT,weights)

## 2. Supply and validate the external artifact

Upload exactly `artifact.json` and `training_context.parquet` from a separate producing execution. Validation occurs before reconstruction and checks model provenance, complete runtime metadata, fitted preprocessing, path/size/digest integrity, symlinks, and unexpected files.

In [ ]:
from pathlib import Path
import shutil
from google.colab import files
A=Path("/content/external-tabdpt-artifact"); shutil.rmtree(A,ignore_errors=True); A.mkdir()
u=files.upload(); expected={"artifact.json","training_context.parquet"}; got={Path(n).name for n in u}
if got!=expected: raise ValueError(f"upload exactly {sorted(expected)}")
for n,b in u.items(): (A/Path(n).name).write_bytes(b)
manifest_path=A/"artifact.json"
manifest,context_path=validate_dimer_artifact(manifest_path,strict_directory=True)
runtime_config=manifest["runtimeConfig"]; preprocessing=manifest["preprocessing"]
print("format/task",manifest["format"],manifest["taskType"],"model",manifest["baseModel"])
print("target/requested drops",runtime_config["target_column"],runtime_config["drop_columns"],"effective fitted drops",preprocessing["dropColumns"])
print("support/validation controls",runtime_config["max_train_rows"],runtime_config["validation_split"])
print("inference controls",{k:runtime_config[k] for k in ("n_ensembles","context_size","batch_size","temperature","seed")})
print("classes/features",manifest["classNames"],preprocessing["encoder"]["featureColumns"])

## 3. Reconstruct the serving state

`load_artifact()` restores fitted preprocessing and support context. Support registration is in-context conditioning, not gradient training; inference data never refits preprocessing.

In [ ]:
pipe=TabDPTClassificationPipeline.load_artifact(manifest_path,compile_model=False,use_flash=False,seed=runtime_config["seed"])
print(pipe.target_column,pipe.class_labels_,pipe.feature_encoder.feature_columns)

## 4. Upload and validate genuinely new input

Upload one external unlabelled CSV not used to create the artifact. The target must be absent and effective features must match the fitted schema exactly.

In [ ]:
import csv, pandas as pd
from collections import Counter
I=Path("/content/tabdpt-input"); shutil.rmtree(I,ignore_errors=True); I.mkdir()
u=files.upload(); names=[Path(n).name for n in u if Path(n).suffix.lower()==".csv"]
if len(u)!=1 or len(names)!=1: raise ValueError("upload exactly one CSV")
p=I/names[0]; p.write_bytes(next(iter(u.values())))
with p.open(encoding="utf-8-sig",newline="") as f: h=next(csv.reader(f),None)
if not h: raise ValueError("empty CSV")
dup=[x for x,n in Counter(h).items() if n>1]
if dup: raise ValueError(f"duplicate columns: {dup}")
new_data=pd.read_csv(p)
if pipe.target_column in new_data: raise ValueError("remove target from inference input")
effective=new_data.drop(columns=pipe.drop_columns_,errors="ignore"); required=list(pipe.feature_encoder.feature_columns)
missing=[x for x in required if x not in effective]; extra=[x for x in effective if x not in required]
if missing or extra: raise ValueError(f"feature schema mismatch missing={missing} extra={extra}")
for col,mapping in pipe.feature_encoder.category_maps.items():
    unseen=sorted({str(v) for v in effective[col].dropna().tolist()}-set(mapping))
    if unseen: print(f"WARNING: {col!r} has unseen categories handled by the fitted unknown-category code: {unseen[:10]}")
print("Validated new input",new_data.shape)

## 5. Predict and export machine-readable results

Serving controls come only from validated artifact metadata. Prediction is argmax over class scores; score calibration is not established.

In [ ]:
import hashlib, json
inference_kwargs = {key: runtime_config[key] for key in ("n_ensembles","context_size","batch_size","temperature","seed")}
pred=pipe.predict(new_data,**inference_kwargs); scores=pipe.predict_proba(new_data,**inference_kwargs)
result=pd.DataFrame({"row_id":new_data.index.astype(str),"prediction":pred.astype(str)})
for label in pipe.class_labels_: result[f"score_{label}"]=scores[label].to_numpy()
O=Path("/content/tabdpt-artifact-inference-output"); O.mkdir(exist_ok=True); result.to_csv(O/"predictions.csv",index=False)
sha=lambda x:hashlib.sha256(Path(x).read_bytes()).hexdigest()
provenance={"profile":"ARTIFACT-INFERENCE","repositoryRevision":REPO_REVISION,"artifact":{"format":manifest["format"],"manifestSha256":sha(manifest_path),"contextSha256":manifest["trainingContext"]["sha256"]},"model":manifest["baseModel"],"runtimeConfig":runtime_config,"classOrder":list(pipe.class_labels_),"input":{"filename":p.name,"sha256":sha(p),"rows":len(new_data)},"runtime":{"python":sys.version.split()[0],"torch":md.version("torch"),"tabdpt":md.version("tabdpt"),"device":torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU","cuda":torch.version.cuda,"use_flash":False,"compile_model":False,"quantization":None,"precision":"framework/device default"},"decisionRule":"argmax","calibrationEstablished":False}
(O/"provenance.json").write_text(json.dumps(provenance,indent=2)); print(result.head(),O)

## Interpretation and troubleshooting

A successful run demonstrates external-artifact validation, serving-state reconstruction, genuinely new-data inference, and machine-readable export. It does **not** prove accuracy, calibration, fairness, robustness, sender authenticity, or production fitness. If provenance, size, digest, or runtime-contract validation fails, do not reconstruct the artifact; obtain a correct artifact from the producer. If feature validation fails, correct the new CSV to match the fitted schema rather than refitting preprocessing. Clean Google Colab execution for the exact release revision remains required; static CI is not execution evidence. A useful next experiment is a separate labelled, domain-valid evaluation and calibration study without changing this artifact-inference notebook's unlabelled-input contract.